In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from openai import OpenAI
import os
from dotenv import load_dotenv
import time

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

def extract_wanted_job_text_selenium(url):
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                         "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36")

    driver = webdriver.Chrome(options=options)
    driver.get(url)
    time.sleep(5)

    wait = WebDriverWait(driver, 20)
    try:
        button = wait.until(
            EC.element_to_be_clickable((By.XPATH, "//button[span[contains(text(), '상세 정보 더 보기')]]"))
        )
        wait.until(EC.visibility_of(button))

        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", button)
        time.sleep(1)

        driver.execute_script("""
            const blocker = document.querySelector('div.WantedApplyBtn_container__lBx_L');
            if (blocker) { blocker.remove(); }
        """)

        time.sleep(0.5)

        driver.execute_script("arguments[0].click();", button)

        wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.JobDescription_JobDescription__paragraph__87w8I"))
        )
        time.sleep(1)
    except Exception as e:
        print("상세 정보 더 보기 버튼 클릭 실패 또는 없음:", e)
        driver.save_screenshot("error_screenshot.png")

    soup = BeautifulSoup(driver.page_source, "html.parser")
    driver.quit()

    containers = soup.find_all('div', class_='JobDescription_JobDescription__paragraph__87w8I')
    if not containers:
        print("채용 공고 본문을 찾지 못했습니다.")
        return None

    texts = [c.get_text(separator='\n', strip=True) for c in containers]
    text = '\n\n'.join(texts)
    return text

def generate_interview_questions(job_text):
    prompt = f"""
다음은 채용 공고 내용입니다. 이 내용을 바탕으로 면접관이 물어볼 수 있는 질문을 5~10개 예상해 주세요.
구체적으로, 업무 수행 능력, 역량, 성향, 지원 동기, 경험 중심의 질문 위주로 작성해 주세요.

[채용 공고]
{job_text}

[예상 면접 질문]
"""
    response = client.chat.completions.create(
        model = "gpt-4o",
        messages = [{"role" : "user", "content" : prompt}],
        temperature = 0.7
    )
    return response.choices[0].message.content.strip()

def main():
    url = input("원티드 채용 공고 URL 입력.").strip()
    print("채용 공고 텍스트 추출 중 ...")
    job_text = extract_wanted_job_text_selenium(url)
    if not job_text:
        print("채용 공고 내용을 찾지 못 했습니다.")
        return

    print("\n추출된 채용 공고 텍스트 일부 :\n")
    print(job_text, "\n")

    print("면접 질문 생성 중 ...")
    questions = generate_interview_questions(job_text)
    print("\n예상 면접 질문 :\n")
    print(questions)

main()

채용 공고 텍스트 추출 중...

추출된 채용공고 텍스트 일부:

주요업무
• 폐기물, 환경자원 흐름에 대한 새로운 사용자 경험(UX) 설계 및 인터페이스(UI) 디자인
• B2B / B2G / B2C 영역을 포괄하는 프로덕트 디자인 전략 수립 및 실행
• 디자인 시스템 및 공통 컴포넌트 개발 및 유지 관리
• 사용자 피드백 기반 개선 및 테스트 주도 (정량적/정성적 방법 병행)
• PO, 개발, 사업부서와의 협업을 통한 기능 기획 및 제품 출시 주도
• 글로벌 확장을 고려한 다국어/다문화 사용자 경험 설계

자격요건
• 플랫폼 혹은 디지털 제품에서의 UI/UX 디자인 경력 2년 이상
• 프로덕트 초기 설계부터 출시까지 전 단계 경험 (특히 MVP 설계 및 스프린트 운영 등)
• Figma 등 협업툴에 대한 능숙한 활용 능력
• 사용자 중심 사고에 기반한 문제 해결 경험
• 기획자/개발자와 유연한 커뮤니케이션을 수행할 수 있는 협업 능력

우대사항
• UX/UI 또는 관련 디자인 전공자
• 디자인 시스템 또는 브랜드 가이드 구축 및 운영 경험
• 데이터 기반 디자인 결정 경험 (A/B 테스트, UX 리서치 등)
• 환경, 공공, O2O 등 복잡한 구조의 서비스 디자인 경험
• 스타트업 혹은 빠른 성장 단계의 조직에서의 실무 경험
• 정부·지자체 대상 서비스 경험 또는 정책 관련 디자인 경험

혜택 및 복지
[기업문화]
• 회사의 운영과 정보는 누구에게나 공평하게 제공합니다
• 직급이 없는 수평문화를 추구하고 누구나 영어닉네임을 사용합니다
• 팀원에게 늘 최고의 보상과 최고의 감동을 드려야 한다고 생각합니다
• 스스로 일을 찾을 수 있는 인재가 회사를 성장시키고 개인도 성장한다고 믿습니다
[인재상]
• 무엇보다 좋은 성향과 품성을 보유하신 분
• 스타트업을 이해하고 그 안에서 자신의 업무를 찾아가는 분
• 자신의 일을 사랑하는 만큼 일을 책임지고 마무리 지을 수 있는 분
[주요복지]
• 폐기물 수거서비스를 비용에 상관없이 무료로 이용할 수 있습니다
• 업무와 관